<a href="https://colab.research.google.com/github/tomarpraveengithub460/PDF2VIDEOAI/blob/main/aipdftovideo_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pypdf

# New section

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pypdf
import nltk

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
file_path = '/content/UNIT I CV.pdf'

try:
    with open(file_path, 'rb') as file:
        reader = pypdf.PdfReader(file)

        # Get the number of pages
        num_pages = len(reader.pages)
        print(f"Number of pages in the PDF: {num_pages}")

        # Iterate through each page and extract text
        full_text = ""
        for page_num in range(num_pages):
            page = reader.pages[page_num]
            text = page.extract_text()
            full_text += f"\n--- Page {page_num + 1} ---\n"
            full_text += text

        print("\nExtracted Text:")
        print(full_text)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please make sure the PDF exists at the specified path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize
import string

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word.isalpha() and word not in stop_words]
    return tokens

print("Preprocessing function defined.")

In [ ]:
sentences = nltk.sent_tokenize(full_text)

In [ ]:
pip install gensim

In [ ]:
preprocessed_sentences = []
for sentence in sentences:
    tokens = preprocess_text(sentence)
    preprocessed_sentences.append(tokens)

print("First 5 preprocessed sentences:")
for i, sent in enumerate(preprocessed_sentences[:5]):
    print(f"Sentence {i+1}: {sent}")

In [ ]:
from gensim.models import Word2Vec
model = Word2Vec(vector_size=100, window=5, min_count=1, workers=4, sg=1)
model.build_vocab(preprocessed_sentences)
model.train(preprocessed_sentences, total_examples=len(preprocessed_sentences), epochs=model.epochs)
print("Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(model.wv)}")

In [ ]:
import numpy as np

def get_sentence_embedding(word_tokens, model):
    embeddings = []
    for word in word_tokens:
        if word in model.wv:
            embeddings.append(model.wv[word])
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(model.vector_size)

sentence_embeddings = []
for preprocessed_sentence in preprocessed_sentences:
    embedding = get_sentence_embedding(preprocessed_sentence, model)
    sentence_embeddings.append(embedding)

sentence_embeddings = np.array(sentence_embeddings)

print("Sentence embeddings generated successfully.")
print(f"Shape of sentence_embeddings: {sentence_embeddings.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = []
for i in range(len(sentence_embeddings) - 1):
    embedding1 = sentence_embeddings[i].reshape(1, -1)
    embedding2 = sentence_embeddings[i+1].reshape(1, -1)
    similarity = cosine_similarity(embedding1, embedding2)[0][0]
    similarities.append(similarity)

threshold = 0.5

topic_boundaries = []
for i, sim in enumerate(similarities):
    if sim < threshold:

        topic_boundaries.append(i + 1)

print("First 10 similarity scores:")
print(similarities[:10])
print("\nIdentified topic boundaries (sentence indices where new topic might start):")
print(topic_boundaries[:20])

In [ ]:
topic_segments = []
current_topic_start_index = 0
for boundary_index in topic_boundaries:
    segment = sentences[current_topic_start_index:boundary_index]
    if segment:
        topic_segments.append(segment)
    current_topic_start_index = boundary_index
if current_topic_start_index < len(sentences):
    final_segment = sentences[current_topic_start_index:]
    if final_segment:
        topic_segments.append(final_segment)
print(f"Total topics identified: {len(topic_segments)}")
print("\n--- Displaying Identified Topic Segments ---")
for i, topic in enumerate(topic_segments):
    print(f"\nTopic {i+1}:")
    for sentence in topic[:5]:
        print(f"- {sentence.strip()}")
    if len(topic) > 5:
        print(f"  ... (total {len(topic)} sentences in this topic)")

In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

print("Model loaded successfully!")

In [ ]:
def rephrase_topic_with_llm(topic_sentences):
    topic_text = "\n".join(topic_sentences)

    prompt = f"""
    Rephrase the following text into 2-3 concise and explainable sentences.
    Make it suitable for an audio recording.
    Include a simple example if helpful.

    Text:
    {topic_text}
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [ ]:
topic_segments

In [ ]:
def contains_reference_or_date(text):

    citation_pattern = r"\[\d+\]"

    author_year_pattern = r"\([A-Za-z\s.,&]+,\s?\d{4}\)"

    year_pattern = r"\b(19|20)\d{2}\b"

    date_pattern = r"\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b"

    if (re.search(citation_pattern, text) or
        re.search(author_year_pattern, text) or
        re.search(year_pattern, text) or
        re.search(date_pattern, text)):
        return True

    return False

In [ ]:
def rephrase_topic_with_llm(topic_sentences):

    processed_sentences = []

    for sentence in topic_sentences:

        if contains_reference_or_date(sentence):
            processed_sentences.append(sentence)
        else:
            prompt = f"""
            Rephrase the following sentence to make it clearer,
            more explainable, and suitable for audio explanation.
            Do NOT change factual meaning.

            Sentence:
            {sentence}
            """

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.7,
                top_p=0.9
            )

            rephrased = tokenizer.decode(outputs[0], skip_special_tokens=True)
            processed_sentences.append(rephrased)

    return " ".join(processed_sentences)

In [ ]:
rephrased_topics = []

print("\n--- Rephrasing Topics with Hugging Face LLM ---")

for i, topic in enumerate(topic_segments):
    print(f"\nRephrasing Topic {i+1}...")

    rephrased_text = rephrase_topic_with_llm(topic)
    rephrased_topics.append(rephrased_text)

    print(f"Original Length (sentences): {len(topic)}")
    print(f"Rephrased Text:\n{rephrased_text}")

print(f"\nSuccessfully rephrased {len(rephrased_topics)} topics.")

In [ ]:
!apt-get update
!apt-get install -y espeak espeak-ng

In [ ]:
pip install gTTS pydub

In [ ]:
from gtts import gTTS
import os
import time

In [ ]:
def save_audio_blocks_online(audio_blocks, output_folder="audio_outputs"):

    os.makedirs(output_folder, exist_ok=True)

    for i, block in enumerate(audio_blocks):

        filename = f"{output_folder}/lecture_part_{i+1}.mp3"

        print(f"Generating {filename}...")

        # Generate speech
        tts = gTTS(text=block, lang='en', slow=False)
        tts.save(filename)

        print(f"Saved: {filename}")

        time.sleep(3)  # Important: avoid rate limit

In [ ]:
def combine_topics_into_15min_blocks(
        rephrased_topics,
        words_per_minute=100,
        target_minutes=15):

    target_word_limit = words_per_minute * target_minutes

    audio_blocks = []
    current_block = []
    current_word_count = 0

    for topic_index, topic in enumerate(rephrased_topics):

        topic_word_count = len(topic.split())

        if topic_word_count >= target_word_limit:

            if current_block:
                audio_blocks.append("\n\n".join(current_block))
                current_block = []
                current_word_count = 0

            audio_blocks.append(topic)
            continue

        if current_word_count + topic_word_count > target_word_limit:

            audio_blocks.append("\n\n".join(current_block))

            current_block = [topic]
            current_word_count = topic_word_count

        else:
            current_block.append(topic)
            current_word_count += topic_word_count

    if current_block:
        audio_blocks.append("\n\n".join(current_block))

    return audio_blocks

In [ ]:
def combine_topics_into_15min_blocks(
        rephrased_topics,
        words_per_minute=100,
        target_minutes=15):

    target_word_limit = words_per_minute * target_minutes

    audio_blocks = []
    current_block = []
    current_word_count = 0

    for topic_index, topic in enumerate(rephrased_topics):

        topic_word_count = len(topic.split())

        if topic_word_count >= target_word_limit:

            if current_block:
                audio_blocks.append("\n\n".join(current_block))
                current_block = []
                current_word_count = 0

            audio_blocks.append(topic)
            continue

        if current_word_count + topic_word_count > target_word_limit:

            audio_blocks.append("\n\n".join(current_block))

            current_block = [topic]
            current_word_count = topic_word_count

        else:
            current_block.append(topic)
            current_word_count += topic_word_count

    if current_block:
        audio_blocks.append("\n\n".join(current_block))

    return audio_blocks

In [ ]:
audio_blocks = combine_topics_into_15min_blocks(rephrased_topics)

print(f"Total audio files: {len(audio_blocks)}")

save_audio_blocks_online(audio_blocks)

In [ ]:
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -r requirements.txt

In [ ]:
!wget "https://storage.googleapis.com/wav2lip/wav2lip_gan.pth"

In [ ]:
!mkdir checkpoints
!mv wav2lip_gan.pth checkpoints/

In [ ]:
!pip uninstall -y numpy librosa

In [ ]:
!pip install numpy==1.23.5
!pip install librosa==0.8.0

In [ ]:
import numpy
print(numpy.__file__)

In [ ]:
numpy_path = "/usr/local/lib/python3.12/dist-packages/numpy/__init__.py"

with open(numpy_path, "a") as f:
    f.write("\ncomplex = complex\n")
    f.write("bool = bool\n")
    f.write("int = int\n")
    f.write("float = float\n")
    f.write("object = object\n")

print("NumPy permanently patched")

In [ ]:
import numpy as np

np.complex = complex
np.bool = bool
np.object = object
np.int = int
np.float = float

print("NumPy patched successfully")

In [ ]:
!wget "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth" -O checkpoints/s3fd.pth

In [ ]:
!wget "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth" -O checkpoints/wav2lip_gan.pth

In [ ]:
import os

audio_folder = "/content/audio_outputs"
lecture_video = "/content/teacher_video.mp4"   # 👈 your original lecture video
output_folder = "/content/wav2lip_results"

os.makedirs(output_folder, exist_ok=True)

%cd /content/Wav2Lip

for file in os.listdir(audio_folder):
    if file.endswith(".mp3"):

        audio_path = os.path.join(audio_folder, file)
        output_path = os.path.join(output_folder, file.replace(".mp3", ".mp4"))

        print(f"\nGenerating lip-synced video for {file}...")

        !python inference.py \
        --checkpoint_path checkpoints/wav2lip_gan.pth \
        --face "{lecture_video}" \
        --audio "{audio_path}" \
        --outfile "{output_path}" \
        --pads 0 20 0 0 \
        --resize_factor 1 \
        --nosmooth

In [ ]:
%cd content

In [ ]:
# Clone repo
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker

# Install dependencies
!pip install -r requirements.txt

# Download pretrained models
!bash scripts/download_models.sh

In [ ]:
!pip install kornia

In [ ]:
!pip install facexlib

In [ ]:
!pip install yacs

In [ ]:
pip uninstall -y gfpgan

In [ ]:
pip install gfpgan==1.3.8

In [ ]:
pip install torch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1

In [ ]:
from pydub import AudioSegment
import os

input_audio = "/content/audio_outputs/lecture_part_1.mp3"
chunk_length = 30000  # 30 seconds

audio = AudioSegment.from_mp3(input_audio)

chunk_folder = "/content/audio_chunks"
os.makedirs(chunk_folder, exist_ok=True)

chunk_paths = []

for i in range(0, len(audio), chunk_length):
    chunk = audio[i:i+chunk_length]
    chunk_path = f"{chunk_folder}/chunk_{i//chunk_length}.mp3"
    chunk.export(chunk_path, format="mp3")
    chunk_paths.append(chunk_path)

print("Created chunks:")
print(chunk_paths)

In [ ]:
%cd /content/SadTalker

In [ ]:
from moviepy.editor import VideoFileClip

input_path = "/content/teacher_video.mp4"
output_path = "/content/teacher_video_fixed.mp4"

clip = VideoFileClip(input_path)

# Force safe encoding
clip = clip.resize((512, 512))  # important
clip.write_videofile(
    output_path,
    codec="libx264",
    audio=False,
    fps=25
)

print("Video fixed and saved.")

In [ ]:
import os
import glob
import subprocess

source_image = "/content/teacher1.png"
results_root = "/content/sadtalker_results"
expression = "/content/teacher_video_fixed.mp4"

os.makedirs(results_root, exist_ok=True)

generated_videos = []

for idx, chunk_path in enumerate(chunk_paths):

    result_dir = f"{results_root}/part_{idx}"
    os.makedirs(result_dir, exist_ok=True)

    command = [
        "python", "inference.py",
        "--driven_audio", chunk_path,
        "--source_image", source_image,
        "--ref_pose", expression,
        "--result_dir", result_dir,
        "--still",
        "--preprocess", "full",
        "--size", "256",
        "--batch_size", "1"
    ]

    print(f"\nProcessing chunk {idx}...")

    process = subprocess.run(command, capture_output=True, text=True)

    # Print real errors if any
    print("STDOUT:\n", process.stdout)
    print("STDERR:\n", process.stderr)

    if process.returncode != 0:
        print(f"❌ Error occurred in chunk {idx}")
        continue

    mp4_files = glob.glob(f"{result_dir}/**/*.mp4", recursive=True)

    if mp4_files:
        generated_videos.append(mp4_files[0])
        print(f"✅ Generated: {mp4_files[0]}")
    else:
        print(f"⚠️ No video found for chunk {idx}")

print("\nAll generated videos:")
print(generated_videos)

In [ ]:
merge_list_path = "/content/merge_list.txt"

with open(merge_list_path, "w") as f:
    for video in generated_videos:
        f.write(f"file '{video}'\n")

final_output = "/content/final_merged_video2.mp4"

!ffmpeg -f concat -safe 0 -i {merge_list_path} -c copy {final_output}

print("\nFinal merged video saved at:")
print(final_output)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install moviepy openai-whisper keybert diffusers transformers accelerate torch torchvision scipy safetensors
!apt-get install ffmpeg -y

In [ ]:
import whisper
import torch
import os
from keybert import KeyBERT
from diffusers import StableDiffusionPipeline
from moviepy.editor import VideoFileClip, ImageClip, CompositeVideoClip, ColorClip

In [ ]:
video_path = "/content/2026_02_24_17.18.12.mp4"

model = whisper.load_model("base")
result = model.transcribe(video_path)

text = result["text"]
print("Full Speech Text:\n")
print(text)

In [ ]:
kw_model = KeyBERT()
keywords = kw_model.extract_keywords(text, top_n=4)

keywords = [kw[0] for kw in keywords]
print("Extracted Keywords:", keywords)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)

os.makedirs("generated_images", exist_ok=True)

for i, word in enumerate(keywords):
    prompt = f"educational diagram of {word}, classroom style, white background, clean illustration"
    image = pipe(prompt).images[0]
    image = image.resize((512, 512))
    image.save(f"generated_images/{i}.jpg")

print("Images Generated Successfully")

In [ ]:
video = VideoFileClip(video_path)

panel_width = video.w // 3

# White side panel background
panel = ColorClip(
    size=(panel_width, video.h),
    color=(245, 245, 245)
).set_duration(video.duration)

clips = [video, panel.set_position(("left","center"))]

duration_per_image = video.duration / len(keywords)

for i in range(len(keywords)):
    img_clip = (
        ImageClip(f"generated_images/{i}.jpg")
        .set_duration(duration_per_image)
        .resize(width=panel_width - 60)
        .set_position((30, "center"))
        .set_start(i * duration_per_image)
        .crossfadein(0.5)
    )
    clips.append(img_clip)

final = CompositeVideoClip(clips)

final.write_videofile("final_output.mp4", codec="libx264", fps=24)

In [ ]:
!pip install moviepy

In [ ]:
from moviepy.editor import *

neutral = VideoFileClip("/content/final_merged_video1.mp4")
pointing = VideoFileClip("/content/final_merged_video2.mp4")

In [ ]:
duration = min(neutral.duration, pointing.duration)

neutral = neutral.subclip(0, duration)
pointing = pointing.subclip(0, duration)

In [ ]:
duration = min(neutral.duration, pointing.duration)

neutral = neutral.subclip(0, duration)
pointing = pointing.subclip(0, duration)

In [ ]:
switch_time = 5
point_duration = 4  # how long hand stays raised

clip1 = neutral.subclip(0, switch_time)

clip2 = pointing.subclip(switch_time, switch_time + point_duration)

clip3 = neutral.subclip(switch_time + point_duration, duration)

final = concatenate_videoclips([clip1, clip2, clip3])

In [ ]:
clip1 = neutral.subclip(0, switch_time)

clip2 = pointing.subclip(switch_time, switch_time + point_duration).crossfadein(0.5)

clip3 = neutral.subclip(switch_time + point_duration, duration).crossfadein(0.5)

final = concatenate_videoclips([clip1, clip2, clip3], method="compose")

In [ ]:
clip2 = pointing.subclip(switch_time, switch_time + point_duration)
clip2 = clip2.resize(lambda t: 1 + 0.02*t).crossfadein(0.5)

In [ ]:
final.write_videofile("teacher_final.mp4", codec="libx264")

In [ ]:
%cd content/

In [ ]:
from moviepy.editor import *

neutral = VideoFileClip("/content/final_merged_video1.mp4")
pointing = VideoFileClip("/content/final_merged_video2.mp4")

# Make sure sizes match
pointing = pointing.resize(neutral.size)

switch_time = 5
transition = 1  # seconds

# First part (neutral)
part1 = neutral.subclip(0, switch_time)

# Transition part
neutral_part = neutral.subclip(switch_time, switch_time + transition)
pointing_part = pointing.subclip(switch_time, switch_time + transition)

cross = neutral_part.crossfadeout(transition)
cross = concatenate_videoclips([cross, pointing_part.crossfadein(transition)], method="compose")

# Remaining pointing
part2 = pointing.subclip(switch_time + transition)

# Combine everything
final = concatenate_videoclips([part1, cross, part2], method="compose")

final.write_videofile("teacher_smooth.mp4", codec="libx264", audio_codec="aac")

In [ ]:
!apt-get install ffmpeg -y

In [ ]:
import subprocess
import math
import os

video_path = "/content/teacher_video.mp4"
audio_path = "/content/lecture_part_1 (2).mp3"
output_path = "/content/final_looped_video.mp4"

def get_duration(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} not found")

    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            file_path
        ],
        capture_output=True,
        text=True
    )

    return float(result.stdout.strip())

# Get durations
video_duration = get_duration(video_path)
audio_duration = get_duration(audio_path)

print("Video Duration:", video_duration)
print("Audio Duration:", audio_duration)

# Calculate loops needed
loops_needed = math.ceil(audio_duration / video_duration)
print("Loops Needed:", loops_needed)

# Create looped video matching audio length exactly
subprocess.run([
    "ffmpeg", "-y",
    "-stream_loop", str(loops_needed - 1),
    "-i", video_path,
    "-i", audio_path,
    "-t", str(audio_duration),
    "-c:v", "libx264",
    "-c:a", "aac",
    "-shortest",
    output_path
])

print("Final video saved at:", output_path)

In [ ]:
!pip install moviepy

In [ ]:
import os
import subprocess

video_path = "/content/final_looped_video (1).mp4"
audio_path = "/content/lecture_part_1 (2).mp3"

chunk_length = 1  # seconds

os.makedirs("/content/chunks/video", exist_ok=True)
os.makedirs("/content/chunks/audio", exist_ok=True)

# Get total video duration
import json

result = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", video_path],
    capture_output=True, text=True
)

duration = float(json.loads(result.stdout)['format']['duration'])
print("Total Duration:", duration)

num_chunks = int(duration // chunk_length) + 1

for i in range(num_chunks):
    start = i * chunk_length

    # Split video
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start),
        "-t", str(chunk_length),
        "-i", video_path,
        "-c", "copy",
        f"/content/chunks/video/video_{i}.mp4"
    ])

    # Split audio
    subprocess.run([
        "ffmpeg", "-y",
        "-ss", str(start),
        "-t", str(chunk_length),
        "-i", audio_path,
        f"/content/chunks/audio/audio_{i}.wav"
    ])

print("Splitting complete.")

In [ ]:
os.makedirs("/content/chunks/frames", exist_ok=True)

video_chunks = sorted(os.listdir("/content/chunks/video"))

for i, file in enumerate(video_chunks):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", f"/content/chunks/video/{file}",
        "-vf", "select=eq(n\\,0)",
        "-q:v", "3",
        f"/content/chunks/frames/frame_{i}.png"
    ])

print("Frame extraction complete.")

In [ ]:
os.makedirs("/content/chunks/output", exist_ok=True)

frame_files = sorted(os.listdir("/content/chunks/frames"))
audio_files = sorted(os.listdir("/content/chunks/audio"))

%cd /content/SadTalker

for i in range(len(frame_files)):

    source_image = f"/content/chunks/frames/frame_{i}.png"
    driven_audio = f"/content/chunks/audio/audio_{i}.wav"
    result_dir = f"/content/chunks/output/output_{i}"

    !python inference.py \
        --driven_audio {driven_audio} \
        --source_image {source_image} \
        --result_dir {result_dir} \
        --still \
        --preprocess full

print("SadTalker generation complete.")

In [ ]:
import shutil

final_clips = []

for i in range(len(frame_files)):
    output_folder = f"/content/chunks/output/output_{i}"
    for file in os.listdir(output_folder):
        if file.endswith(".mp4"):
            src = os.path.join(output_folder, file)
            dst = f"/content/chunks/output/clip_{i}.mp4"
            shutil.copy(src, dst)
            final_clips.append(dst)

print("Collected generated clips.")

In [ ]:
with open("/content/chunks/output/list.txt", "w") as f:
    for clip in sorted(final_clips):
        f.write(f"file '{clip}'\n")

subprocess.run([
    "ffmpeg", "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", "/content/chunks/output/list.txt",
    "-c", "copy",
    "/content/final_lecture.mp4"
])

print("Final video created at /content/final_lecture.mp4")